# PurushTradingAIAgents — train the PPO position sizer (Colab/Kaggle, free GPU)

The sizer only chooses a risk fraction in [0,1] per candidate trade; direction, entries and hard rule caps come from the pipeline. It is registered with an out-of-sample comparison against fixed-fraction sizing and is **used only if it beat it** (`beats_fixed == 1`). Replace `YOUR_GITHUB_USER`.

> Paper research only. Not investment advice.

In [ ]:
!pip -q install "git+https://github.com/YOUR_GITHUB_USER/PurushTradingAIAgents.git#egg=ptaa" stable-baselines3 torch
import os, pathlib
DATA = pathlib.Path('/content/drive/MyDrive/ptaa-data')  # mount Drive first (see train_gbm_colab.ipynb)
os.environ['PTAA_DATA_DIR'] = str(DATA)
!curl -sO https://raw.githubusercontent.com/YOUR_GITHUB_USER/PurushTradingAIAgents/main/config.yaml
from ptaa.settings import load_config
cfg = load_config('config.yaml')

In [ ]:
# 1. Build the sizing episode from a trained GBM's out-of-fold probabilities
from ptaa.mlmodels.registry import ModelRegistry
from ptaa.training.pipeline import run_training, yfinance_candle_source
from ptaa.training.paper_env import episode_from_backtest
reg = ModelRegistry(DATA / 'models')
source = yfinance_candle_source(DATA, years=12)
# re-run training here to get meta + OOF probabilities in memory (fast on CPU)
from ptaa.training.dataset import build_dataset
from ptaa.training.labels import BarrierConfig
from ptaa.training.train_gbm import train_gbm
from ptaa.data.universe import NIFTY50
ds = build_dataset(NIFTY50, source, BarrierConfig(horizon=5))
res = train_gbm(ds, reg, horizon='5d', universe='NIFTY50', min_train=20000)
ep = episode_from_backtest(ds.meta, res.oof_proba)
len(ep.score)

In [ ]:
# 2. Train PPO and export the deterministic actor to ONNX
from ptaa.training.train_ppo import train_ppo
out = train_ppo(ep, cfg.risk['swing'], cfg.fees, reg, universe='NIFTY50', timesteps=300_000)
out

## Reading the result

`ppo_final_equity` vs `fixed_final_equity` and `half_final_equity` on the held-out 20% of candidates, with drawdowns. `beats_fixed` must be 1.0 for the trader node to use the policy; otherwise the fixed-fraction sizer stays in place. Copy `models/ppo/*.onnx` and `models/registry.json` to the laptop (GitHub release assets).